In [ ]:
import sys

from numba import njit
import numba as nb
import numpy as np
import pickle as pkl

from scipy.ndimage import label, generate_binary_structure, find_objects

---
---
---
---
---
---
---
---
---
---

In [ ]:
@njit#(parallel=True)
def voids_edge_coords_fct(fg_100, i1, size):

    '''
    Void cells that neighbour walls + void cells on the edge
    '''

    voids_edge_coords_i1 = np.argwhere(fg_100 == i1)
    
    len_vec = len(voids_edge_coords_i1)
    voids_edge_coords_i = [[-1,-1,-1] for _ in range(len_vec)]

    u0 = -1
    for veii0 in range(len_vec):
        u0 += 1

        veii = voids_edge_coords_i1[veii0]
        
        if ((veii[0] in [0, size-1]) or (veii[1] in [0, size-1]) or (veii[2] in [0, size-1])):
            voids_edge_coords_i[u0] = list(veii); continue
        
        touches_wall = False
        for i in range(-1,2):
            for j in range(-1,2):
                for k in range(-1,2):
                    ii = (veii[0]+i)%size
                    jj = (veii[1]+j)%size
                    kk = (veii[2]+k)%size
                    if (fg_100[ii][jj][kk] == -2):
                        voids_edge_coords_i[u0] = list(veii)
                        touches_wall = True; break
                if touches_wall: break
            if touches_wall: break

    return [_ for _ in voids_edge_coords_i if _[0] != -1]

In [ ]:
def get_patch_bounds(labeled_array, no_patches):
    
    
    locations = find_objects(labeled_array, max_label=no_patches)
    bounds = []
    
    for i in range(no_patches):
        sl_x, sl_y, sl_z = locations[i]
        x_min = sl_x.start; x_max = sl_x.stop-1
        y_min = sl_y.start; y_max = sl_y.stop-1
        z_min = sl_z.start; z_max = sl_z.stop-1
        bounds.append([[x_min, x_max], [y_min, y_max], [z_min, z_max]])
    
    return bounds

In [ ]:
def find_engulfed_patches(bounds, labeled_array, no_patches, labeled_array1, size):
    
    engulfed_patches = np.zeros(no_patches, dtype=bool)
    
    for i in range(no_patches):
        for j in range(no_patches):
            if j == i: continue
            
            bounds_i = bounds[i]; bounds_j = bounds[j]
            is_engulfed = True
            mm_ij_match = [0][:0]
            
            for dim in range(3):
                min_i, max_i = bounds_i[dim]
                min_j, max_j = bounds_j[dim]
                if (min_i == min_j == 0) or (max_i == max_j == size-1): mm_ij_match.append(dim)
                
                if (min_j > 0)      and (min_i <= min_j): is_engulfed = False; break
                if (max_j < size-1) and (max_i >= max_j): is_engulfed = False; break


            # If we haven't found the patch to be not engulfed but both it and the other patch do touch at least one same edge, then it may still
            #    be that patch j does extend the most in every other direction, but it still does not engulf i.
            # Again, if at least one coordinate of i is greater than j's, then ofc j cannot engulf it. But if at least one of both ends on the edge,
            #    then we know these can be separate patches just starting from at least that edge!
            # Because of that, we need to check if between them fg_100 is the same void or a wall or another void. This is the simplest and fastest test!
            # It may be that they have multiple which they touch (i might be in the corner).
            
            # Now that we know we can find them on one side and that, if anything, it is i that must be engulfed, not j (since its max on any coordinate is
            #    either larger or equal, the later only at edges), we can simply check if they share the same patch formed for the full void patches (not just
            #    its edges).
            # If they do, i was in fact engulfed.
            # If they don't then it is a separate patch.
            if is_engulfed and len(mm_ij_match) != 0:
                dim = mm_ij_match[0]
                min_i, max_i = bounds_i[dim]
                min_j, max_j = bounds_j[dim]
                start_coord = 0
                if   min_i == min_j == 0:      start_coord = 0
                elif max_i == max_j == size-1: start_coord = size-1

                if   dim == 0:
                    labeled_array_edge  = labeled_array[ start_coord, :, :]
                    labeled_array_edge1 = labeled_array1[start_coord, :, :]
                elif dim == 1:
                    labeled_array_edge  = labeled_array[ :, start_coord, :]
                    labeled_array_edge1 = labeled_array1[:, start_coord, :]
                else:         
                    labeled_array_edge  = labeled_array[ :, :, start_coord]
                    labeled_array_edge1 = labeled_array1[:, :, start_coord]

                i_coord = np.argwhere(labeled_array_edge == i+1)[0]
                j_coord = np.argwhere(labeled_array_edge == j+1)[0]
                
                if labeled_array_edge1[i_coord[0]][i_coord[1]] != labeled_array_edge1[j_coord[0]][j_coord[1]]: is_engulfed = False
                
            
            if is_engulfed: engulfed_patches[i] = True; break
    
    return engulfed_patches

---

In [ ]:
def check_this_patch(p_i, labeled_array, no_patches, size):

    '''
    We find all the connections to patch p_i and the directions from it to them.
    '''
    
    BAD_void = False
    connections = np.zeros(no_patches, dtype=bool)
    directions  = [np.array([0,0,0][:0]) for _ in range(no_patches)]

    coords_p_i = np.argwhere(labeled_array == p_i)
    edge_coords_p_i = coords_p_i[np.any(np.isin(coords_p_i, [0, size-1]), axis=1)]
    
    for edge_coord in edge_coords_p_i:
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                for dz in [-1, 0, 1]:
                    if dx != 0 or dy != 0 or dz != 0:
                        
                        ii0, ii1, ii2 = [(edge_coord[0] + dx) % size, (edge_coord[1] + dy) % size, (edge_coord[2] + dz) % size]
                        
                        direction = np.array([1 if (dx == 1 and edge_coord[0] == size-1) else -1 if (dx == -1 and edge_coord[0] == 0) else 0,
                                              1 if (dy == 1 and edge_coord[1] == size-1) else -1 if (dy == -1 and edge_coord[1] == 0) else 0,
                                              1 if (dz == 1 and edge_coord[2] == size-1) else -1 if (dz == -1 and edge_coord[2] == 0) else 0])
                        
                        if np.any(direction):
                            p_j = labeled_array[ii0][ii1][ii2]
                            if p_j != -1:
                                if len(directions[p_j]) == 0: connections[p_j] = True; directions[p_j] = direction
                                elif not np.array_equal(directions[p_j], direction): BAD_void = True; break
                                if p_j == p_i:                                       BAD_void = True; break
                if BAD_void: break
            if BAD_void: break
        if BAD_void: break
    
    return connections, directions, BAD_void

---

In [ ]:
offsets = np.array([[-1, -1, -1], [-1, -1,  0], [-1, -1,  1],
                    [-1,  0, -1], [-1,  0,  0], [-1,  0,  1], 
                    [-1,  1, -1], [-1,  1,  0], [-1,  1,  1],
                    [ 0, -1, -1], [ 0, -1,  0], [ 0, -1,  1],
                    [ 0,  0, -1],               [ 0,  0,  1],
                    [ 0,  1, -1], [ 0,  1,  0], [ 0,  1,  1],
                    [ 1, -1, -1], [ 1, -1,  0], [ 1, -1,  1],
                    [ 1,  0, -1], [ 1,  0,  0], [ 1,  0,  1],
                    [ 1,  1, -1], [ 1,  1,  0], [ 1,  1,  1]])

In [ ]:
@njit
def check_neighbor_walls_cube(fg_100, cell, size):

    for i,j,k in (cell + offsets) % size:
        if fg_100[i][j][k] == -2: return True
    
    return False

---
---
---